# ClaimWise Insurance Prediction — 10. Final Prediction

## 1. Objective

Demonstrate the complete ClaimWise prediction workflow end to end:

1. load the comparison results and the final trained model,
2. prepare input data exactly like the training pipeline does,
3. generate a prediction of the insurance claim loss,
4. show and explain the predicted value.

**Input data:** the sample rows used below are real rows from the project's held-out test set
(`Dataset/06_split_X_test.csv`). They are clearly labelled as sample/test input — no customer
information is invented anywhere in this notebook.

## 2. Import Libraries

In [1]:
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root  :", PROJECT_ROOT)
print("Dataset folder:", DATASET_DIR)
print("Src folder    :", SRC_DIR)

from model_training_utils import (
    calculate_metrics,
    load_prepared_data,
    save_model_results,
)

Project root  : /Users/padaltiruvinayak/Desktop/Credit ML
Dataset folder: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset
Src folder    : /Users/padaltiruvinayak/Desktop/Credit ML/Src


## 3. Load the Final Model

The best model is the one selected by `09_ClaimWise_Model_Comparison.ipynb` (lowest RMSE) and saved
to `Models/best_model.pkl`. If those files are not present yet, the full training pipeline is run
first so this notebook always works from a clean state.

In [3]:
comparison_path = OUTPUTS_DIR / "Model_Comparison" / "model_comparison.csv"
best_model_path = MODELS_DIR / "best_model.pkl"

if not (comparison_path.exists() and best_model_path.exists()):
    print("Comparison or best model not found - running the full training pipeline first...")
    from train_models import main as train_all_models
    train_all_models()

comparison = pd.read_csv(comparison_path)
display(comparison)

best_name = str(comparison.iloc[0]["Model"])
best_metrics = comparison.iloc[0][["MAE", "MSE", "RMSE", "R2"]].to_dict()
print("Final model selected for prediction:", best_name)
print("Its test metrics:", best_metrics)

,Model,MAE,MSE,RMSE,R2
0,Hist Gradient Boosting Regressor,1215.299703,3.690333e+06,1921.023976,0.545901
1,Random Forest Regressor,1259.621849,3.937887e+06,1984.410916,0.515440
2,Gradient Boosting Regressor,1277.782914,4.021054e+06,2005.256485,0.505206
3,Linear Regression,1333.566472,4.204642e+06,2050.522301,0.482615
4,Decision Tree Regressor,1609.918901,6.964030e+06,2638.944950,0.143070


Final model selected for prediction: Hist Gradient Boosting Regressor
Its test metrics: {'MAE': 1215.299702959891, 'MSE': 3690333.1164736343, 'RMSE': 1921.023976027794, 'R2': 0.5459011973814277}


In [4]:
best_model = joblib.load(best_model_path)

print("Loaded model :", best_model_path)
print("Model class  :", type(best_model).__name__)
params = pd.Series(best_model.get_params(), name="value").to_frame()
display(params)

Loaded model : /Users/padaltiruvinayak/Desktop/Credit ML/Models/best_model.pkl
Model class  : HistGradientBoostingRegressor


,value
categorical_features,from_dtype
early_stopping,auto
interaction_cst,None
l2_regularization,0.5
learning_rate,0.035
loss,squared_error
max_bins,255
max_depth,None
max_features,0.8
max_iter,830


## 4. Prepare Input Data

The prepared test features are loaded with the same validation helper used during training, so the
prediction input has exactly the 130 features the model expects.

In [5]:
X_train, X_test, y_train, y_test = load_prepared_data()

print("Prediction input shape:", X_test.shape)
print("Features expected by the model:", X_test.shape[1])
print("First 3 feature names:", list(X_test.columns[:3]))

Prediction input shape: (10000, 130)
Features expected by the model: 130
First 3 feature names: ['cat1', 'cat2', 'cat3']


## 5. Sample Prediction (labelled test input)

Five real rows from the held-out test set are passed to the model. The predicted loss is compared
with the actual loss that was recorded for those rows.

In [6]:
n_samples = 5
sample_input = X_test.head(n_samples)
sample_actual = y_test.head(n_samples)
sample_predicted = best_model.predict(sample_input)

sample_results = pd.DataFrame({
    "sample_test_row": range(1, n_samples + 1),
    "actual_loss": sample_actual.to_numpy(),
    "predicted_loss": np.round(sample_predicted, 2),
    "absolute_error": np.round(np.abs(sample_actual.to_numpy() - sample_predicted), 2),
})
print("Sample / test input predictions:")
display(sample_results)

Sample / test input predictions:


,sample_test_row,actual_loss,predicted_loss,absolute_error
0,1,3302.33,3565.33,263.00
1,2,1021.53,2061.51,1039.98
2,3,580.00,4287.41,3707.41
3,4,1109.79,2042.22,932.43
4,5,4325.72,3473.08,852.64


## 6. Prediction on the Whole Test Set

Generating predictions for all 10,000 test rows and re-checking the metrics of the saved model.

In [7]:
y_pred_all = best_model.predict(X_test)

test_results = pd.DataFrame({
    "actual_loss": y_test.to_numpy(),
    "predicted_loss": np.round(y_pred_all, 2),
    "residual": np.round(y_test.to_numpy() - y_pred_all, 2),
})

FINAL_PREDICTION_DIR = OUTPUTS_DIR / "Final_Prediction"
FINAL_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

sample_path = FINAL_PREDICTION_DIR / "claimwise_sample_predictions.csv"
full_path = FINAL_PREDICTION_DIR / "claimwise_test_set_predictions.csv"
sample_results.to_csv(sample_path, index=False)
test_results.to_csv(full_path, index=False)

metrics = calculate_metrics(y_test, y_pred_all)
print("Metrics of the final model on the test set:")
display(pd.DataFrame([{"Model": best_name, **metrics}]))
print("Saved:", sample_path)
print("Saved:", full_path)

Metrics of the final model on the test set:


,Model,MAE,MSE,RMSE,R2
0,Hist Gradient Boosting Regressor,1215.299703,3.690333e+06,1921.023976,0.545901


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/Final_Prediction/claimwise_sample_predictions.csv
Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/Final_Prediction/claimwise_test_set_predictions.csv


## 7. What the Output Represents

- `predicted_loss` is the model's estimate of the insurance **claim loss amount** for that row,
  expressed in the same unit and range as the original `loss` column.
- `actual_loss` is the true value that was kept aside during training; it is used only to measure
  how close the prediction is.
- `residual = actual_loss - predicted_loss`: a positive value means the model under-predicted the
  loss, a negative value means it over-predicted.
- The overall quality of the predictions is summarised by MAE / MSE / RMSE / R² printed above:
  RMSE is the typical size of the error in the same unit as `loss`, and R² is the fraction of the
  variation in `loss` the model explains.

In [8]:
print("Test rows predicted        :", len(test_results))
print("Mean actual loss (test set) :", round(float(test_results['actual_loss'].mean()), 2))
print("Mean predicted loss         :", round(float(test_results['predicted_loss'].mean()), 2))
print("Mean absolute error (MAE)   :", round(metrics['MAE'], 4))
print("RMSE                        :", round(metrics['RMSE'], 4))
print("R2                          :", round(metrics['R2'], 6))

Test rows predicted        : 10000
Mean actual loss (test set) : 3009.45
Mean predicted loss         : 3002.62
Mean absolute error (MAE)   : 1215.2997
RMSE                        : 1921.024
R2                          : 0.545901


## 8. Conclusion

The complete ClaimWise workflow — raw dataset → preprocessing → train/test split → trained model →
prediction — ran successfully in this notebook using only the project's own files.

Results of this run:

- Final model: **Hist Gradient Boosting Regressor** (`Models/best_model.pkl`)
- Sample predictions (5 real held-out test rows) saved to
  `Outputs/Final_Prediction/claimwise_sample_predictions.csv`
- All 10,000 test predictions saved to
  `Outputs/Final_Prediction/claimwise_test_set_predictions.csv`
- Test-set metrics of the final model: MAE 1215.2997, MSE 3690333.1165, RMSE 1921.0240, R² 0.545901
- Mean actual loss 3009.45 vs mean predicted loss 3002.62 on the test set

Every number above was produced by executing this notebook; nothing was typed in by hand.